# **01 LangChain 기초**

### 학습 내용
1. Chat Models 초기화
2. Messages 이해
3. 대화 히스토리 관리
4. 스트리밍 응답

## 0. 환경 설정

- OpenAI API Key 발급: https://platform.openai.com/api-keys

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("✓ API Key가 설정되었습니다.")

✓ API Key가 설정되었습니다.


## 1. Chat Models(LLM) 초기화

### `init_chat_model()`

참고: https://docs.langchain.com/oss/python/langchain/models

In [5]:
# 방법 1: init_chat_model() 사용
from langchain.chat_models import init_chat_model

# 모델 초기화
llm = init_chat_model(
    "gpt-5.4-mini",  # 또는 "gpt-4o", "claude-sonnet-4-5-20250929"
)

# 모델 호출
response = llm.invoke("LangChain이란 무엇인지 3문장으로 설명해줘.")
print(response.content)

LangChain은 대규모 언어 모델(LLM)을 이용한 애플리케이션을 쉽게 만들 수 있도록 도와주는 오픈소스 프레임워크입니다.  
프롬프트 관리, 외부 데이터 연결, 도구 호출, 여러 단계의 작업 흐름(에이전트) 구성 같은 기능을 제공합니다.  
즉, LLM을 단순히 “질문-답변”에 쓰는 것을 넘어 실제 서비스나 자동화 도구로 확장할 수 있게 해줍니다.


### 주요 파라미터

| 파라미터 | 설명 | 기본값 |
|---------|------|--------|
| `model` | 사용할 모델 이름 (예: "gpt-4o-mini") | 필수 |
| `temperature` | 창의성 조절 (0.0=결정적, 1.0=창의적) | 모델별 상이 |
| `max_tokens` | 생성할 최대 토큰 수 | 모델별 상이 |
| `timeout` | API 요청 타임아웃 (초) | 없음 |
| `max_retries` | 실패 시 재시도 횟수 (지수 백오프 적용) | 6 |

## 2. 메시지 (Messages) 이해

**메시지(Message)** 는 LangChain에서 대화를 관리하는 핵심 개념입니다.

### 2-1. 메시지 구조

1. **Role (역할)** - 메시지를 보낸 주체 (`system`, `user`, `assistant`)
2. **Content (내용)** - 메시지 텍스트
3. **Metadata (메타데이터)** - 추가 정보 (ID, 타임스탬프, 토큰 사용량 등)

In [6]:
conversation = [
    {"role": "system", "content": "당신은 한국어를 프랑스어로 번역하는 번역가입니다."},
    {"role": "user", "content": "번역: 나는 프로그래밍을 좋아합니다."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "번역: 나는 애플리케이션을 만드는 것을 좋아합니다."}
]

response = llm.invoke(conversation)
print(response)  # AIMessage("J'adore créer des applications.")

content="J'aime créer des applications." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 70, 'total_tokens': 79, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EE5NV7f84uvGZmhek32LwyTtLASCu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0130c-5e81-7e72-9d1f-6515424efe65-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 70, 'output_tokens': 9, 'total_tokens': 79, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [5]:
response.content

"J'aime créer des applications."

### 2-2. 문자열 vs 메시지 객체

| 항목 | 문자열 | 메시지 객체 |
|-----|--------------|---------------|
| 사용법 | 간단함 | Message 클래스 사용 |
| 역할 지정 | 기본 1개만 | 여러 역할, 메타데이터 가능 |
| 적합한 경우 | 단순 질문 | 복잡한 대화 |

참고: https://docs.langchain.com/oss/python/langchain/messages

**문자열 사용 예시:**
```python
response = model.invoke("Write a haiku about spring")
```

**메시지 객체 사용 예시:**
```python
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a haiku about spring"),
    AIMessage("Cherry blossoms bloom...")
]
response = model.invoke(messages)
```

### 2-3. 메시지 타입 (Message Types)


**1) SystemMessage - 시스템 메시지**
- 모델의 행동을 정의합니다. 모델에게 역할을 부여합니다.

**2) HumanMessage - 사용자 메시지**
- 사용자의 입력입니다. 질문, 요청, 명령 등을 나타냅니다.

**3) AIMessage - AI 응답**
- 모델의 응답입니다. 추가로 tool_calls, usage_metadata 등을 포함할 수 있습니다.

**4) ToolMessage - 도구 결과**
- Tool 실행 결과를 나타냅니다.

In [7]:
# 문자열 사용 (간단한 방법)
response = llm.invoke("AI가 무엇인가요?")
print("=== 문자열 사용 ===")
print(response.content)
print(f"응답 타입: {type(response)}")

=== 문자열 사용 ===
AI는 **인공지능(Artificial Intelligence)**의 약자입니다.  
사람이 하던 **학습, 판단, 인식, 언어 이해, 문제 해결** 같은 일을 컴퓨터가 하도록 만든 기술을 말합니다.

예를 들면:
- 음성 인식: “헤이 시리”, “오케이 구글”
- 추천 시스템: 유튜브, 넷플릭스 추천
- 챗봇: 질문에 답하는 AI
- 이미지 인식: 사진 속 사람이나 사물 구분

쉽게 말해, **데이터를 바탕으로 스스로 패턴을 배우고 작업을 수행하는 똑똑한 프로그램**이라고 볼 수 있습니다.

원하시면 제가 **AI의 종류**, **작동 원리**, 또는 **일상 속 AI 예시**도 쉽게 설명해드릴게요.
응답 타입: <class 'langchain_core.messages.ai.AIMessage'>


In [8]:
# 메시지 객체 사용 (자세한 방법)
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage("당신은 친절한 AI 어시스턴트입니다."),
    HumanMessage("AI가 무엇인가요?")
]
response = llm.invoke(messages)
print("=== 메시지 객체 사용 ===")
print(response.content)
print(f"응답 타입: {type(response)}")

=== 메시지 객체 사용 ===
AI는 **인공지능(Artificial Intelligence)**의 줄임말로,  
사람처럼 **학습하고 판단하고 문제를 해결하는 컴퓨터 기술**을 말합니다.

쉽게 말하면, AI는 컴퓨터가:

- **데이터를 보고 패턴을 배우고**
- **질문에 답하거나**
- **이미지/음성을 인식하고**
- **추천을 해주거나**
- **자동으로 작업을 수행**할 수 있게 해주는 기술입니다.

예를 들면:
- 음성 비서
- 번역기
- 영화/상품 추천
- 얼굴 인식
- 챗봇

원하시면 제가 **AI의 종류**, **작동 방식**, 또는 **일상 속 AI 예시**도 쉽게 설명해드릴게요.
응답 타입: <class 'langchain_core.messages.ai.AIMessage'>


In [8]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# SystemMessage로 역할 정의 - 종료 시까지 유지됨
system_msg = SystemMessage("""
당신은 10년 경력의 Python 개발자입니다.
사용자의 질문에 정확하고 친절하게 답변하세요.
코드 예시를 포함하세요.
""")

messages = [
    system_msg,
    HumanMessage(content="Python에서 리스트 컴프리헨션을 설명해주세요.")
]

response = llm.invoke(messages)
print(response.content)
print()

리스트 컴프리헨션(List Comprehension)은 **리스트를 간결하게 만드는 문법**입니다.  
반복문과 조건문을 한 줄에 압축해서 새로운 리스트를 만들 수 있어요.

---

## 1) 기본 형태

```python
[표현식 for 변수 in 반복가능한객체]
```

예를 들면, 0부터 4까지 숫자의 제곱 리스트를 만들 수 있습니다.

```python
squares = [x * x for x in range(5)]
print(squares)
# [0, 1, 4, 9, 16]
```

위 코드는 아래와 같은 `for`문과 같은 의미입니다.

```python
squares = []
for x in range(5):
    squares.append(x * x)
```

---

## 2) 조건문 포함하기

조건을 넣어서 특정 값만 골라낼 수 있습니다.

```python
evens = [x for x in range(10) if x % 2 == 0]
print(evens)
# [0, 2, 4, 6, 8]
```

이것도 일반 `for`문으로 쓰면 다음과 같습니다.

```python
evens = []
for x in range(10):
    if x % 2 == 0:
        evens.append(x)
```

---

## 3) 조건에 따라 값 바꾸기

리스트 컴프리헨션 안에서 `if ... else ...`를 사용해 값을 바꿀 수도 있습니다.

```python
result = ["짝수" if x % 2 == 0 else "홀수" for x in range(5)]
print(result)
# ['짝수', '홀수', '짝수', '홀수', '짝수']
```

주의할 점은 이 경우 문법이:

```python
[참일때값 if 조건 else 거짓일때값 for 변수 in 반복가능한객체]
```

형태라는 것입니다.

---

## 4) 중첩 for문도 가능

```python
pairs = [(x, y) for x in range(2) for y

### 📖 과제 1: 시스템 프롬프트로 챗봇의 역할 정의하기

SystemMessage로 시스템 프롬프트(페르소나, 역할, 규칙 등 정의)를 작성하고 답변이 어떻게 달라지는 지 확인해봅시다.

다양한 도메인의 전문가로 AI의 역할을 다시 정의하거나, 더 나은 답변을 출력하기 위해 시스템 프롬프트를 바꿔보세요.

- 주제 (선생님 / 의사 / 소설가 / 요리사 / 법률 상담가 등)
- 말투 (친근하게 / 엄격하게 / 비유 많이 사용)
- 답변 형식 (항상 단계별 / 항상 예시 포함 / 항상 해시태그 포함)
- 대상 독자 (초등학생 / 비전공자 / 전문가)


In [11]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# TODO: SystemMessage 작성
system_msg = SystemMessage("""
    주제 코딩, 말투 매우 매우 예의 없는 교사(욕설 사용 교사), 대상 독자 코딩 입문하려는 학생
""")

messages = [
    system_msg,
    HumanMessage(content="선생님 오버로딩과 오버라이딩에 대해서 차이점과 활용 예시를 상세히 설명해주세요")
]

response = llm.invoke(messages)
print(response.content)
print()

좋다, 이걸 헷갈리면 안 된다. **오버로딩(Overloading)**이랑 **오버라이딩(Overriding)**은 이름만 비슷하지, **완전히 다른 개념**이다. 코딩 입문자라면 이 둘을 딱 구분해두는 게 중요하다. 헷갈리면 나중에 클래스 설계할 때 아주 골치 아프다.  

---

# 1. 한 줄로 먼저 구분

- **오버로딩**:  
  **같은 이름의 메서드/함수를 여러 개 만드는 것**  
  단, **매개변수 목록이 달라야 함**

- **오버라이딩**:  
  **부모 클래스에 있는 메서드를 자식 클래스가 다시 정의하는 것**  
  즉, **상속 관계에서 메서드 동작을 바꾸는 것**

---

# 2. 오버로딩(Overloading)

## 개념
같은 이름의 메서드를 여러 개 만들 수 있는데,  
대신 **매개변수의 개수, 타입, 순서** 중 하나가 달라야 한다.

즉, 이름은 같아도 **받는 입력이 다르면 다른 메서드처럼 취급**하는 거다.

---

## 예시 1: 자바(Java)에서 오버로딩

```java
class Calculator {
    int add(int a, int b) {
        return a + b;
    }

    int add(int a, int b, int c) {
        return a + b + c;
    }

    double add(double a, double b) {
        return a + b;
    }
}
```

### 설명
- `add(int, int)` → 2개 정수 더하기
- `add(int, int, int)` → 3개 정수 더하기
- `add(double, double)` → 실수 2개 더하기

이렇게 **이름은 add로 같지만 입력 형태가 다르므로** 오버로딩이다.

---

## 오버로딩을 왜 쓰나?
### 1) 함수 이름을 의미 있게 통일할 수 있음
예를 들어 숫자 덧셈은 전부 `add`로 묶고 싶을 때 좋다.

### 2) 사용자가 편함
```java

## 3. 대화 히스토리 관리

Chat Model은 이전 대화를 기억하지 않습니다. 필요한 대화 내용은 개발자가 직접 전달해야 합니다.

### 3-1. 대화 관리 흐름 이해하기

1. 사용자 메시지를 HumanMessage로 변환
2. 모델에 전달
3. 응답을 AIMessage로 저장
4. 다음 요청 시 이전 대화를 함께 전달

In [10]:
# 대화 히스토리 축적
chat_history = []

# 첫 번째 대화
print("=== 대화 1 ===")
user_msg_1 = HumanMessage(content="내 이름은 찰리야.")
chat_history.append(user_msg_1)

response_1 = llm.invoke(chat_history)
print(f"User: {user_msg_1.content}")
print(f"AI: {response_1.content}")

# AI 응답 저장
chat_history.append(response_1)
print(f"\n대화 히스토리: {len(chat_history)}개 메시지")
for i, msg in enumerate(chat_history, 1):
    print(f"{i}. [{msg.type}] {msg.content}")

=== 대화 1 ===
User: 내 이름은 찰리야.
AI: 반가워, 찰리!  
무엇을 도와줄까?

대화 히스토리: 2개 메시지
1. [human] 내 이름은 찰리야.
2. [ai] 반가워, 찰리!  
무엇을 도와줄까?


In [11]:
# 두 번째 대화
print("\n=== 대화 2 ===")
user_msg_2 = HumanMessage(content="내 이름이 뭐였어?")
chat_history.append(user_msg_2)

response_2 = llm.invoke(chat_history)
print(f"User: {user_msg_2.content}")
print(f"AI: {response_2.content}")
chat_history.append(response_2)

# 히스토리 확인
print(f"\n대화 히스토리: {len(chat_history)}개 메시지")
for i, msg in enumerate(chat_history, 1):
    print(f"{i}. [{msg.type}] {msg.content}")


=== 대화 2 ===
User: 내 이름이 뭐였어?
AI: 네 이름은 찰리야.

대화 히스토리: 4개 메시지
1. [human] 내 이름은 찰리야.
2. [ai] 반가워, 찰리!  
무엇을 도와줄까?
3. [human] 내 이름이 뭐였어?
4. [ai] 네 이름은 찰리야.


### 3-2. 멀티턴 동작

> **멀티턴(Multi-turn)** 은 AI나 챗봇이 이전 대화의 맥락과 이력을 기억한 채 여러 번 질문과 답변을 주고받는 상호작용 방식을 말합니다.

다음은 멀티턴을 기반으로 동작하는 간단한 챗봇입니다. 직접 대화를 나눠보며, 챗봇이 대화를 잘 기억하는 지 확인해봅시다.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from IPython.display import Markdown, display

conversation = [
    SystemMessage(content="당신은 여행 가이드입니다. 여행 장소에 대한 여행 계획이나 참고할만한 정보를 제공하세요."),
]

print("\n🧭 여행 가이드 챗봇 시작\n")
print("-" * 50)

while True:
    user_input = input("🧑 User: ")
    print(f"🧑 User: : {user_input}")
    if user_input.lower() in ["q", "exit", "quit"]:
        print("\n👋 대화를 종료합니다.")
        break

    # user 메시지 저장
    conversation.append(HumanMessage(content=user_input))

    response = llm.invoke(conversation)

    # AI 메시지 저장
    conversation.append(AIMessage(content=response.content))

    # 출력 (Markdown 렌더링)
    print("\n🤖 AI:")
    display(Markdown(response.content))
    print("-" * 50)


🧭 여행 가이드 챗봇 시작

--------------------------------------------------
🧑 User: : 일본 갈거야 계획 짜봐

🤖 AI:


좋아요. 일본 여행 계획 짜드릴게요.  
먼저 **무난하고 만족도 높은 5일 일정**으로 예시를 드리고, 원하시면 **도쿄/오사카/교토/후쿠오카** 중 한 곳에 맞춰 더 자세히 맞춤 짜드릴 수 있어요.

## 일본 5일 여행 기본 계획 예시
### 1일차: 도착 + 시내 적응
- 공항 도착
- 숙소 체크인
- 근처 상점가나 번화가 산책
- 현지 음식 먹기: 라멘, 돈카츠, 초밥, 이자카야

### 2일차: 대표 관광지
- 아침: 유명 사원/신사 방문
- 낮: 시내 중심가 쇼핑
- 저녁: 전망대나 야경 명소
- 밤: 현지 맛집 탐방

### 3일차: 근교 이동 또는 테마 일정
- 선택 1: 자연 풍경 보러 가기
- 선택 2: 온천 체험
- 선택 3: 디즈니랜드/유니버셜 스튜디오 같은 테마파크

### 4일차: 문화 + 쇼핑
- 전통 거리, 박물관, 시장 방문
- 기념품 쇼핑
- 카페 투어
- 저녁엔 맛집 중심으로 마무리

### 5일차: 마지막 일정 + 귀국
- 공항 가기 전 간단한 쇼핑
- 라스트 식사
- 귀국

---

## 일본 여행 준비 팁
- **교통**: 지하철/JR/버스 이용이 많아서 교통패스 확인
- **결제**: 카드 가능하지만 현금도 조금 필요
- **통신**: eSIM, 유심, 포켓와이파이 중 선택
- **숙소**: 역 근처가 이동 편함
- **예산**: 도시마다 다르지만 식비/교통비/입장료 고려 필요

---

## 추천 여행 스타일
- **처음 일본 가는 경우**: 도쿄 3~5일
- **먹거리+쇼핑 중심**: 오사카 3~4일
- **전통+풍경**: 교토 포함 일정
- **짧고 편한 여행**: 후쿠오카
- **휴양/온천**: 삿포로, 하코네, 벳푸

원하시면 제가 바로 아래 중 하나로 **구체적인 일정** 짜드릴게요.

1. **도쿄 3박 4일**
2. **오사카/교토 4박 5일**
3. **후쿠오카 2박 3일**
4. **일본 첫 여행용 추천 코스**
5. **가성비 여행 코스**

가고 싶은 도시랑 여행 기간 알려주시면 딱 맞게 짜드릴게요.

--------------------------------------------------
🧑 User: : 도쿄 일주일

🤖 AI:


좋아요. **도쿄 1주일 여행** 기준으로, 처음 가도 무리 없고 관광/쇼핑/맛집/근교까지 균형 있게 넣은 일정으로 짜드릴게요.

---

# 도쿄 7일 여행 계획

## 1일차: 도착 + 시부야/신주쿠 적응
**추천 일정**
- 나리타/하네다 공항 도착
- 숙소 체크인
- **시부야 스크램블 교차로**
- **하치코 동상**
- **시부야 스카이** 야경
- 저녁: 시부야 맛집 or 이자카야

**포인트**
- 첫날은 무리하지 말고 시내 중심으로
- 밤에 야경 보면 도쿄 분위기 바로 느낄 수 있어요

---

## 2일차: 아사쿠사 + 우에노 + 스카이트리
**추천 일정**
- **센소지 절**
- 나카미세 거리 산책
- 전통 간식 먹기
- **우에노 공원**
- 우에노 아메요코 시장 구경
- 저녁: **도쿄 스카이트리** 또는 주변 식사

**포인트**
- 도쿄의 전통 분위기를 느끼는 날
- 기념품 사기 좋음

---

## 3일차: 하라주쿠 + 오모테산도 + 신주쿠
**추천 일정**
- **메이지 신궁**
- **하라주쿠 다케시타 거리**
- 오모테산도 카페 투어
- **신주쿠 교엔**
- 저녁: 신주쿠 가부키초, 도쿄 도청 전망대

**포인트**
- 젊은 분위기, 쇼핑, 카페, 도심 풍경을 한 번에 즐기기 좋음

---

## 4일차: 디즈니랜드 또는 디즈니씨
**추천 일정**
- 하루 종일 **도쿄 디즈니랜드**
  또는
- 어른 취향이면 **도쿄 디즈니씨**

**포인트**
- 도쿄 여행에서 만족도 높은 날
- 미리 입장권 예매 추천
- 아침 일찍 가야 많이 즐길 수 있어요

---

## 5일차: 오다이바 + 도쿄만 주변
**추천 일정**
- **오다이바**
- 팀랩 플래닛(운영 여부 확인 필요)
- 다이버시티 건담
- 레인보우 브리지 주변
- 해변 산책
- 쇼핑몰 구경

**포인트**
- 미래도시 느낌 나는 지역
- 실내/실외 조합이라 편하게 다니기 좋음

---

## 6일차: 근교 여행
### 선택 1: 하코네
- 온천, 자연, 후지산 뷰
- 편안하게 쉬는 일정

### 선택 2: 가마쿠라
- **대불**, 사찰, 바다 풍경
- 느긋한 감성 여행

### 선택 3: 요코하마
- 차이나타운
- 대관람차
- 항구 야경

**추천**
- 첫 도쿄 여행이면 **가마쿠라**나 **요코하마**가 부담 적어요

---

## 7일차: 마지막 쇼핑 + 귀국
**추천 일정**
- 숙소 주변 카페
- 돈키호테, 유니클로, 무지, 백화점 등 쇼핑
- 공항으로 이동
- 귀국

**포인트**
- 마지막 날은 너무 빡빡하게 잡지 않는 게 좋아요

---

# 도쿄 여행 추천 지역별 숙소
- **시부야**: 젊고 활기찬 분위기, 교통 편함
- **신주쿠**: 교통 최강, 맛집/쇼핑 많음
- **우에노**: 상대적으로 가성비 좋음
- **긴자**: 깔끔하고 고급스러움
- **아사쿠사**: 전통 분위기, 비교적 조용함

---

# 교통 팁
- 도쿄는 **지하철 + JR**가 핵심
- 숙소는 **환승 적은 역 근처**가 편함
- 모바일 **교통카드(Suica/PASMO)** 쓰면 편리
- 공항 이동은 **게이큐/스카이라이너/리무진버스** 확인

---

# 맛집 추천 카테고리
- 라멘
- 규카츠
- 스시
- 돈카츠
- 야키토리
- 편의점 음식도 꼭 한번 체험

---

# 일주일 예산 감각
대략적으로:
- **절약형**: 70만~100만 원대
- **보통형**: 100만~180만 원대
- **여유형**: 180만 원 이상

※ 항공권 시기, 숙소 위치, 디즈니/근교 일정에 따라 크게 달라져요.

---

원하시면 제가 다음 중 하나로 바로 더 구체적으로 짜드릴게요.

1. **도쿄 7일 상세 시간표**
2. **가성비 도쿄 7일**
3. **쇼핑 위주 도쿄 7일**
4. **맛집 위주 도쿄 7일**
5. **혼자 가는 도쿄 7일**
6. **커플/가족용 도쿄 7일**

원하시는 스타일 말해주시면 그에 맞춰 완성해드릴게요.

--------------------------------------------------
🧑 User: : 호텔명 까지 매우 상세히 계획 짜줘

🤖 AI:


좋아요. 다만 **호텔명을 “확정”해서 짜려면 예산, 여행 시기, 공항(나리타/하네다), 동행자 수**가 중요해요.  
그래서 우선은 **도쿄 7일 여행용 “실행 가능한 예시안”**으로, **지역별 추천 호텔명까지 포함한 매우 상세 계획**을 드릴게요.  
아래 일정은 **처음 도쿄 가는 분 기준, 이동 편하고 관광 효율 좋은 코스**입니다.

---

# 도쿄 7일 매우 상세 여행 계획
## 컨셉
- **1~3일차:** 도쿄 도심 관광
- **4일차:** 디즈니 또는 대형 테마 일정
- **5일차:** 오다이바/도쿄만
- **6일차:** 근교
- **7일차:** 쇼핑 후 귀국

---

# 숙소 추천: 3개 호텔명 예시
도쿄는 **숙소를 1곳만 잡아도 되고**, 더 효율적으로는 **2곳으로 나눠도 좋습니다.**

## 옵션 A. 숙소 1곳만 사용
### 추천 호텔: **Hotel Gracery Shinjuku**
- 위치: 신주쿠 가부키초
- 장점: 교통 좋음, 식당 많음, 밤에도 편리
- 단점: 번화가라 조용한 분위기는 아님

**이런 분에게 추천**
- 관광 동선 편하게 하고 싶은 경우
- 밤까지 돌아다니는 일정이 많은 경우

---

## 옵션 B. 가성비 + 이동 편의
### 추천 호텔: **Sotetsu Fresa Inn Tokyo-Kinshicho**
- 위치: 긴시초
- 장점: 비교적 가격 합리적, 조용함, JR/지하철 이용 편함
- 단점: 핵심 관광지까지 100% 도보권은 아님

---

## 옵션 C. 깔끔하고 교통 편한 중급
### 추천 호텔: **Mitsui Garden Hotel Ueno**
- 위치: 우에노
- 장점: 공항 접근성 좋음, 관광 이동 편함, 주변 편의시설 많음
- 단점: 인기 많아 예약 빨리 필요

---

# 상세 일정

## 1일차: 도착 + 신주쿠 적응
### 숙소
- **Hotel Gracery Shinjuku** 체크인 추천

### 일정
- 공항 도착
- 입국 수속
- 숙소 이동
- 체크인 후 짐 정리
- **신주쿠 역 주변 산책**
- **도쿄 도청 전망대**
- 저녁: 신주쿠 맛집
- 밤: 가부키초/오모이데 요코초 가볍게 구경

### 식사 추천
- 라멘
- 규동
- 이자카야

### 이동 팁
- 나리타 도착 시: 공항철도 + 환승
- 하네다 도착 시: 전철/모노레일로 더 편함

---

## 2일차: 시부야 + 하라주쿠 + 오모테산도
### 일정
- 아침: 호텔 조식 또는 근처 카페
- 오전: **시부야 스크램블 교차로**
- **하치코 동상**
- **시부야 스카이** 입장
- 점심: 시부야 돈카츠/덮밥
- 오후: **메이지 신궁**
- **하라주쿠 다케시타 거리**
- **오모테산도** 카페 투어
- 저녁: 시부야 쇼핑 후 귀가

### 포인트
- 이 날은 도보 + 지하철 중심
- 사진 찍기 좋은 장소 많음

### 숙소 귀환
- 신주쿠 숙소면 이동 편리

---

## 3일차: 아사쿠사 + 우에노 + 아키하바라
### 일정
- 오전: **센소지 절**
- **나카미세 거리** 먹거리 탐방
- 점심: 아사쿠사 전통 음식 또는 텐동
- 오후: **우에노 공원**
- **아메요코 시장**
- 시간이 되면 **아키하바라** 이동
- 저녁: 아키하바라 전자상가/애니메이션 거리
- 귀가

### 포인트
- 전통 + 시장 + 오타쿠 문화까지 한 번에 체험

---

## 4일차: 디즈니랜드 or 디즈니씨
### 추천 숙소 전략
- 숙소는 그대로 신주쿠 유지 가능
- 대신 아침 일찍 출발

### 일정
- 오전: 디즈니 도착
- 하루 종일 테마파크
- 밤 퍼레이드/쇼 관람
- 저녁 늦게 귀가

### 팁
- **디즈니랜드**: 가족/처음 가는 사람에게 무난
- **디즈니씨**: 어른 취향, 분위기 좋음

### 준비
- 입장권 사전 예약
- 편한 신발 필수
- 간단한 간식/보조배터리 준비

---

## 5일차: 오다이바 + 팀랩/전망/쇼핑
### 일정
- 오전: 오다이바 이동
- **다이버시티 도쿄 플라자**
- 건담 관람
- **팀랩 플래닛** 또는 유사 전시
- 점심: 해변 근처 쇼핑몰 식당
- 오후: 해변 산책
- **아쿠아시티 오다이바**
- 저녁: 레인보우 브리지 야경

### 포인트
- 실내 비중이 높아서 날씨 영향을 덜 받음

---

## 6일차: 근교 여행
여기서는 취향에 따라 선택하세요.

### 선택 A: 가마쿠라
**추천 이유:** 바다, 사찰, 여유로운 분위기

**일정**
- 오전: 도쿄 출발
- **가마쿠라 대불**
- **하세데라**
- **츠루가오카 하치만구**
- **에노덴 전철** 체험
- 바다 쪽 산책
- 저녁 도쿄 복귀

### 선택 B: 요코하마
**추천 이유:** 이동 부담 적고 도시 야경 좋음

**일정**
- **미나토미라이**
- **컵누들 뮤지엄**
- **요코하마 차이나타운**
- 야경 감상 후 복귀

### 선택 C: 하코네
**추천 이유:** 온천, 휴식, 풍경

**일정**
- 로프웨이, 호수, 온천
- 단, 당일치기보다 1박이 더 좋음

---

## 7일차: 마지막 쇼핑 + 귀국
### 일정
- 체크아웃
- 짐 보관
- **돈키호테**
- **유니클로/무지**
- 기념품 쇼핑
- 공항 이동
- 출국

### 공항 이동 팁
- 나리타: 여유 있게 2~3시간 전 이동
- 하네다: 상대적으로 빠르게 이동 가능

---

# 호텔별로 더 추천해드리면
## 1. 가장 무난한 1순위
### **Hotel Gracery Shinjuku**
- 관광, 맛집, 야간 이동 모두 편함
- 도쿄 초행자에게 가장 안정적

## 2. 깔끔하고 관광 중심
### **Mitsui Garden Hotel Ueno**
- 우에노/아사쿠사 쪽 일정이 많으면 좋음
- 공항 접근성도 괜찮음

## 3. 가성비 중시
### **Sotetsu Fresa Inn**
- 체인 호텔이라 실패 확률 낮음
- 역 근처 지점이 많아서 편리

---

# 추천 최종 조합
## 초행자 + 관광 위주
- 숙소: **Hotel Gracery Shinjuku**
- 주요 지역: 시부야, 신주쿠, 아사쿠사, 우에노, 오다이바
- 근교: 가마쿠라

## 조금 더 조용하고 교통 좋은 곳
- 숙소: **Mitsui Garden Hotel Ueno**
- 주요 지역: 아사쿠사, 우에노, 아키하바라, 긴자
- 근교: 요코하마

---

# 예상 예산
## 1인 기준 대략
- 숙소: 1박 10만~25만 원대부터 다양
- 식비: 하루 3만~8만 원
- 교통비: 하루 1만~2만 원 정도
- 입장료: 디즈니/전망대/전시 별도

---

원하시면 다음 단계로 바로 해드릴 수 있어요:
1. **항공권 포함 전체 예산표**
2. **나리타/하네다 도착 기준 동선**
3. **호텔 3~5개 더 비교**
4. **“아침 몇 시 출발, 몇 시 도착” 수준의 초상세 일정표**
5. **혼자 여행 / 커플 / 가족 맞춤 일정**

원하시면 제가 바로 **“하네다 도착 기준 7일 초상세 시간표”**로 이어서 짜드릴게요.

--------------------------------------------------
🧑 User: : 초호화 여행

🤖 AI:


좋아요. **도쿄 초호화 7일 여행**으로 짜드릴게요.  
컨셉은 **최상급 호텔 + 프리미엄 레스토랑 + 고급 쇼핑 + 편안한 이동**입니다.  
이 일정은 “많이 보는 여행”보다 **좋은 곳에서 여유 있게 즐기는 여행**에 맞춰 구성할게요.

---

# 도쿄 초호화 7일 여행 플랜
## 여행 스타일
- 숙소: 5성급 호텔
- 교통: 택시/공항리무진/프라이빗 이동 위주
- 식사: 미쉐린, 오마카세, 파인다이닝
- 일정: 빡빡하지 않게
- 쇼핑: 긴자, 오모테산도, 이세탄, 도버스트리트 등

---

# 추천 호텔 3곳
## 1. **Aman Tokyo**
- 위치: 오테마치
- 특징: 도쿄 최고급급, 전망과 서비스 압도적
- 분위기: 매우 조용하고 고급스러움
- 추천도: 최상

## 2. **The Peninsula Tokyo**
- 위치: 유라쿠초/긴자 인근
- 특징: 긴자 쇼핑과 식사에 최적
- 분위기: 우아하고 클래식함
- 추천도: 매우 높음

## 3. **The Okura Tokyo**
- 위치: 아카사카
- 특징: 일본식 고급미 + 훌륭한 서비스
- 분위기: 차분하고 품격 있음
- 추천도: 고급 휴식형

---

# 7일 상세 일정

## 1일차: 도착 + 호텔 체크인 + 가볍게 긴자
### 호텔
- **Aman Tokyo** 또는 **The Peninsula Tokyo** 체크인

### 일정
- 공항 도착
- **프라이빗 픽업** 또는 리무진 이동
- 호텔 체크인
- 객실 휴식, 스파 예약 확인
- 저녁: **긴자 고급 스시/가이세키**
- 식후: 긴자 야간 산책
- 호텔 복귀

### 추천 포인트
- 첫날은 절대 빡빡하게 안 움직임
- 호텔 자체를 여행처럼 즐기기

---

## 2일차: 긴자 + 고급 쇼핑 + 미쉐린 디너
### 일정
- 아침: 호텔 조식
- 오전: **긴자 명품 거리 쇼핑**
- 점심: 프렌치/일식 런치 코스
- 오후: **니혼바시**, 백화점, 갤러리
- 티타임: 고급 카페/호텔 라운지
- 저녁: **미쉐린 스타 레스토랑**
- 밤: 호텔 바 or 스카이라운지

### 쇼핑 추천
- Ginza Six
- Mitsukoshi
- Matsuya Ginza
- 유니크한 일본 브랜드 편집숍

---

## 3일차: 아트 + 스파 + 프라이빗한 도쿄
### 일정
- 아침: 여유롭게 시작
- 오전: **모리 미술관** 또는 **도쿄 국립신미술관**
- 점심: 호텔 근처 파인다이닝
- 오후: 호텔 스파 / 마사지
- 저녁: **와규 오마카세**
- 늦은 밤: 호텔 라운지에서 칵테일

### 포인트
- 관광보다 체험의 질을 높이는 날
- 몸을 쉬게 하는 일정

---

## 4일차: 하코네 당일치기 또는 1박
### 옵션 A: 당일치기 하코네
- 프라이빗 차량 또는 럭셔리 교통
- 호수/온천/경치 감상
- 고급 료칸 점심

### 옵션 B: 하코네 1박
- 더 추천
- **고급 료칸 + 개인 노천탕**
- 온천과 자연으로 완전 휴식

### 추천
- 초호화 여행이라면 **하코네 1박**이 가장 만족도 높음

---

## 5일차: 도쿄 복귀 + 오모테산도/하라주쿠 프리미엄 쇼핑
### 일정
- 아침: 온천 후 조식
- 도쿄 복귀
- 호텔 체크인
- 오후: **오모테산도**
- **아오야마** 갤러리/편집숍
- 저녁: 파인다이닝 또는 이탈리안 코스
- 밤: 라운지 바

### 포인트
- 젊고 세련된 럭셔리 무드

---

## 6일차: 맞춤형 럭셔리 체험
### 선택지
#### 1) 요트 크루즈
- 도쿄만 프라이빗 크루즈
- 샴페인 서비스 포함 가능

#### 2) 프라이빗 가이드 투어
- 아사쿠사, 우에노, 메이지신궁 등
- 일반 관광보다 편하고 깊이 있게

#### 3) 최고급 스파 데이
- 호텔 스파 + 페이셜 + 마사지
- 하루 쉬면서 컨디션 회복

### 저녁
- 최종 만찬: **최상급 스시 오마카세**
- 또는 **철판구이 와규 코스**

---

## 7일차: 마지막 쇼핑 + 귀국
### 일정
- 호텔 조식
- 체크아웃
- 짐 맡기기
- **이세탄 신주쿠** 또는 **긴자 백화점**
- 면세 쇼핑
- 공항 이동
- 귀국

---

# 초호화 여행 추천 식사 스타일
## 스시
- 고급 오마카세
- 예약 필수

## 가이세키
- 일본 전통 고급 코스 요리

## 와규
- 철판구이/야키니쿠 프리미엄 코스

## 프렌치/이탈리안
- 도쿄는 세계 수준의 파인다이닝이 많음

---

# 초호화 이동 팁
- 공항: **프라이빗 픽업**
- 도심: **택시 중심**
- 근교: **전용차량** 고려
- 짐 많으면 호텔 포터 서비스 적극 활용

---

# 럭셔리 쇼핑 추천 구역
- **긴자**
- **오모테산도**
- **아오야마**
- **신주쿠 이세탄**
- **도버 스트리트 마켓 긴자**
- **롯폰기 힐즈**

---

# 예상 예산 감각
초호화 기준으로는 크게 차이 나지만 보통:
- 호텔: 1박 수십만~수백만 원
- 식사: 1인 코스당 고가 가능
- 이동: 전용차량/택시 포함
- 쇼핑: 개인 성향에 따라 무제한

---

# 추천 최종 조합
## 가장 럭셔리한 조합
- **호텔: Aman Tokyo**
- **식사: 긴자 스시 + 와규 + 가이세키**
- **근교: 하코네 1박**
- **쇼핑: 긴자 + 오모테산도**

## 우아하고 클래식한 조합
- **호텔: The Peninsula Tokyo**
- **식사: 미쉐린 + 호텔 라운지**
- **일정: 긴자 중심 + 아트 + 스파**

---

원하시면 다음 중 하나로 바로 이어서 아주 구체적으로 짜드릴게요.

1. **초호화 7일을 시간 단위로 세분화**
2. **호텔 3곳 비교표**
3. **예산별 초호화 플랜**
4. **커플용 초호화 도쿄**
5. **1인 VIP 초호화 도쿄**
6. **예약해야 할 식당/스파/체험 리스트**

원하시면 제가 바로 **“Aman Tokyo 기준 초호화 시간표”**로 완성해드릴게요.

--------------------------------------------------


## 4. 스트리밍 응답 (Streaming)

### 4-1. AIMessageChunk

- `invoke()`는 전체 `AIMessage`를 한번에 반환합니다
- `stream()`은 토큰단위로 `AIMessageChunk`를 순차적으로 반환합니다
- 실시간 응답 표시가 가능합니다
- 사용자 경험을 개선합니다

In [13]:
for chunk in llm.stream("Python에 대해 설명해주세요."):
    # print(chunk)
    print(chunk.text, end="", flush=True)
print("\n")

Python은 **읽기 쉽고 배우기 쉬운 범용 프로그래밍 언어**입니다.  
웹 개발, 데이터 분석, 인공지능, 자동화, 과학 계산 등 아주 다양한 분야에서 널리 사용됩니다.

## Python의 특징
- **문법이 간단함**: 다른 언어보다 코드가 짧고 직관적입니다.
- **가독성이 좋음**: 사람이 읽기 쉽게 설계되었습니다.
- **범용성**: 웹, 앱, 게임, 데이터 처리, AI 등 여러 분야에서 활용됩니다.
- **풍부한 라이브러리**: 필요한 기능을 쉽게 가져다 쓸 수 있습니다.
- **활발한 커뮤니티**: 자료가 많고 문제 해결이 비교적 쉽습니다.

## Python 예시
```python
print("Hello, world!")
```

이 코드는 화면에 `Hello, world!`를 출력합니다.

## 간단한 변수 예시
```python
name = "철수"
age = 20

print(name)
print(age)
```

## Python이 많이 쓰이는 분야
- **웹 개발**: Django, Flask
- **데이터 분석**: Pandas, NumPy
- **인공지능/머신러닝**: TensorFlow, PyTorch
- **자동화**: 반복 작업을 자동으로 처리
- **교육용 언어**: 프로그래밍 입문에 적합

## 장점
- 배우기 쉬움
- 빠르게 개발 가능
- 다양한 분야에 적용 가능

## 단점
- C/C++ 같은 언어보다 실행 속도가 느릴 수 있음
- 모바일 앱이나 초고성능 시스템에는 덜 자주 쓰임

원하시면 제가 다음 중 하나로 이어서 설명해드릴 수 있습니다:
1. **Python 문법 기초**
2. **Python 설치 방법**
3. **Python으로 할 수 있는 일**
4. **Python 예제 코드 더 보기**



### 4-2. AIMessageChunk 결합

여러 개의 청크를 합치면 AIMessage와 동일한 구조가 됩니다. `+` 연산자로 `invoke()`처럼 전체 메시지를 복원할 수 있습니다.

In [14]:
full_message = None

for chunk in llm.stream("인공지능이 무엇인가요?"):
    # 청크 결합
    full_message = chunk if full_message is None else full_message + chunk
    print(chunk.text, end="", flush=True)

인공지능(AI)은 **사람처럼 배우고, 판단하고, 문제를 해결하도록 만든 컴퓨터 기술**입니다.

쉽게 말하면:
- **데이터를 보고 패턴을 찾고**
- **그 패턴을 바탕으로 예측하거나 결정하고**
- **대화, 번역, 이미지 인식, 추천** 같은 일을 할 수 있습니다.

예를 들면:
- 음성 비서가 말을 알아듣는 것
- 유튜브/넷플릭스가 영상을 추천하는 것
- 사진 속 얼굴이나 물체를 찾는 것
- 챗봇이 질문에 답하는 것

인공지능은 크게 두 가지로 볼 수 있습니다:
1. **좁은 인공지능(약한 AI)**: 특정 작업만 잘하는 AI  
   - 예: 번역기, 추천 시스템
2. **일반 인공지능(강한 AI)**: 인간처럼 다양한 일을 폭넓게 수행하는 AI  
   - 현재는 아직 완전히 구현되지 않았습니다.

원하시면 제가 **“인공지능이 작동하는 방식”**도 아주 쉽게 설명해드릴게요.

In [15]:
print(f"\n메시지 타입: {type(full_message)}")
print(f"전체 content: {full_message.content}")


메시지 타입: <class 'langchain_core.messages.ai.AIMessageChunk'>
전체 content: 인공지능(AI)은 **사람처럼 배우고, 판단하고, 문제를 해결하도록 만든 컴퓨터 기술**입니다.

쉽게 말하면:
- **데이터를 보고 패턴을 찾고**
- **그 패턴을 바탕으로 예측하거나 결정하고**
- **대화, 번역, 이미지 인식, 추천** 같은 일을 할 수 있습니다.

예를 들면:
- 음성 비서가 말을 알아듣는 것
- 유튜브/넷플릭스가 영상을 추천하는 것
- 사진 속 얼굴이나 물체를 찾는 것
- 챗봇이 질문에 답하는 것

인공지능은 크게 두 가지로 볼 수 있습니다:
1. **좁은 인공지능(약한 AI)**: 특정 작업만 잘하는 AI  
   - 예: 번역기, 추천 시스템
2. **일반 인공지능(강한 AI)**: 인간처럼 다양한 일을 폭넓게 수행하는 AI  
   - 현재는 아직 완전히 구현되지 않았습니다.

원하시면 제가 **“인공지능이 작동하는 방식”**도 아주 쉽게 설명해드릴게요.


### 📖 과제 2: 멀티턴 대화 스트리밍으로 긴 설명 받기

멀티턴 대화 방식으로 AI 응답을 스트리밍 방식(streaming)으로 출력하도록 구현하세요.

1. 멀티턴 대화 유지
- 이전 대화 내용을 conversation 리스트로 관리할 것
- 사용자 입력과 AI 응답이 모두 누적되어야 함
2. 스트리밍 응답 처리
- llm.invoke() 대신 llm.stream() 사용
- AI 응답을 토큰 단위로 실시간 출력할 것
3. 응답 저장
- 스트리밍으로 출력된 내용을 하나의 문자열로 합쳐서
- AIMessage로 conversation에 저장할 것


💡 힌트: 3-2. 멀티턴 에서 진행한 코드를 베이스로 사용하세요!

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

conversation = [
    SystemMessage(content="당신은 여행 가이드입니다. 여행 장소에 대한 여행 계획이나 참고할만한 정보를 제공하세요."),
]

print("\n🧭 여행 가이드 챗봇 시작\n")
print("-" * 50)

while True:
    user_input = input("🧑 User: ")
    print(f"🧑 User: {user_input}")
    
    if user_input.lower() in ["q", "exit", "quit"]:
        print("\n👋 대화를 종료합니다.")
        break

    conversation.append(HumanMessage(content=user_input))

    print("\n🤖 AI: ", end="", flush=True)
    
    full_response = ""
    for chunk in llm.stream(conversation):
        token = chunk.content
        print(token, end="", flush=True)
        full_response += token

    print("\n" + "-" * 50)
    conversation.append(AIMessage(content=full_response))


🧭 여행 가이드 챗봇 시작 (스트리밍 모드)

--------------------------------------------------
🧑 User: 일본 여행 일정 토쿄 7일 이단 150만원선 호텔, 즐길 장소명 등 상세히 일정을 짜줄 것

🤖 AI: 일본 도쿄에서의 7일 여행 일정을 아래와 같이 제안드립니다. 예산은 약 150만원 선으로 설정하였으며, 호텔, 즐길 장소, 교통편 등을 포함했습니다.

### **여행 일정**

#### **1일차: 도쿄 도착**
- **호텔 체크인**: 신주쿠 지역의 중급 호텔 (예: APA Hotel Shinjuku Gyoenmae) - 약 10만 원/박
- **오후**: 신주쿠 탐방
  - 신주쿠 교엔 (Shinjuku Gyoen) - 일본 전통 정원
  - 오모이데 요코초 (Omoide Yokocho)에서 저녁 식사 (일본 전통 꼬치구이)
- **저녁**: 도쿄 도심 야경 감상 (도쿄 메트로폴리탄 정부 청사 전망대 무료)

#### **2일차: 아사쿠사 및 우에노**
- **오전**: 아사쿠사 탐방
  - 센소지 (Senso-ji) 사원 방문
  - 나카미세 거리에서 기념품 쇼핑과 간식
- **점심**: 아사쿠사 지역의 우동집
- **오후**: 우에노 공원 탐방
  - 우에노 동물원 또는 일본국립박물관 방문
- **저녁**: 우에노 지역의 이자카야에서 저녁 식사

#### **3일차: 시부야 및 하라주쿠**
- **오전**: 시부야 탐방
  - 시부야 스크램블 교차로
  - 하치코 동상
- **점심**: 시부야의 유명한 패스트푸드나 카페
- **오후**: 하라주쿠 탐방
  - 다케시타 거리에서 쇼핑
  - 메이지 신궁 방문
- **저녁**: 하라주쿠 지역의 카페에서 디저트

#### **4일차: 도쿄 디즈니랜드 또는 디즈니씨**
- **종일**: 도쿄 디즈니랜드 또는 디즈니씨에서 하루 종일 즐기기
- **저녁**: 테마파크 내에서 저녁 식사 (예: 캐릭터 식사)

#### **5일차: 오다이바 탐방**
- **오전*

---

### 참고 자료

- [LangChain Models 공식 문서](https://docs.langchain.com/oss/python/langchain/models)
- [LangChain Messages 공식 문서](https://docs.langchain.com/oss/python/langchain/messages)
- [Chat Models 통합 문서](https://docs.langchain.com/oss/python/integrations/chat)